In [1]:
# task a.2
import cv2
import numpy as np

def detect_corner_markers(image):
    """
    Find the 4 black square calibration markers.
    Returns: corners array sorted [TL, TR, BR, BL]
    """

    # brg --> hsv
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # threshold 
    _, binary = cv2.threshold(
        hsv[:, :, 2],
        50,
        255,
        cv2.THRESH_BINARY_INV
    )

    # external contours
    contours, _ = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # filter by size & squareness
    square_candidates = []

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area < 100:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        box_area = w * h
        squareness = area / box_area if box_area > 0 else 0

        if squareness > 0.3:
            square_candidates.append(cnt)

    # largest 4
    square_candidates.sort(key=cv2.contourArea, reverse=True)
    top4 = square_candidates[:4]

    if len(top4) < 4:
        raise ValueError("Could not detect 4 corner markers")

    # centroids
    centers = []

    for cnt in top4:
        M = cv2.moments(cnt)

        if M['m00'] == 0:
            continue

        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])

        centers.append((cx, cy))

    # sort
    centers = sort_clockwise(centers)

    return np.float32(centers)


def sort_clockwise(pts):
  
    pts = sorted(pts, key=lambda p: p[1])

    top = sorted(pts[:2], key=lambda p: p[0])
    bottom = sorted(pts[2:], key=lambda p: p[0], reverse=True)

    return [top[0], top[1], bottom[0], bottom[1]]


# load image
image_path = r"C:\Users\Asus\Desktop\hasti\output.png"   # change to your image path
image = cv2.imread(image_path)

if image is None:
    raise FileNotFoundError(f"Image not found: {image_path}")

# detect markers
corners = detect_corner_markers(image)

print("Detected corners:")
print(corners)

# draw detected points
output = image.copy()

labels = ["TL", "TR", "BR", "BL"]

for i, (x, y) in enumerate(corners.astype(int)):
    cv2.circle(output, (x, y), 10, (0, 0, 255), -1)

    cv2.putText(
        output,
        labels[i],
        (x + 10, y - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

# save & show
cv2.imwrite("detected_markers.jpg", output)
cv2.imshow("Detected Markers", output)
cv2.waitKey(0)
cv2.destroyAllWindows()


Detected corners:
[[127. 106.]
 [498.  62.]
 [520. 439.]
 [105. 455.]]


In [2]:
# a.3 (rectification only)
canvas_size = 600

blank = np.ones((canvas_size, canvas_size, 3), dtype=np.uint8) * 255

dst_points = np.float32([
    [0, 0],
    [599, 0],
    [599, 599],
    [0, 599]
])

H, _ = cv2.findHomography(corners, dst_points)

warped = cv2.warpPerspective(
    image,
    H,
    (canvas_size, canvas_size)
)

grid_img = warped.copy()

spacing = 60
dash_length = 10
gap_length = 10

color = (0, 0, 0)
thickness = 1

for x in range(0, canvas_size, spacing):

    y = 0

    while y < canvas_size:

        y_end = min(y + dash_length, canvas_size)

        cv2.line(
            grid_img,
            (x, y),
            (x, y_end),
            color,
            thickness
        )

        y += dash_length + gap_length

for y in range(0, canvas_size, spacing):

    x = 0

    while x < canvas_size:

        x_end = min(x + dash_length, canvas_size)

        cv2.line(
            grid_img,
            (x, y),
            (x_end, y),
            color,
            thickness
        )

        x += dash_length + gap_length

cv2.imwrite("a3.png", grid_img)

cv2.imshow("A3 Output", grid_img)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [3]:
# a.3 (main)
original_display = image.copy()

# center
p1 = corners[0]
p2 = corners[2]

p3 = corners[1]
p4 = corners[3]

x1, y1 = p1.astype(np.float64)
x2, y2 = p2.astype(np.float64)

x3, y3 = p3.astype(np.float64)
x4, y4 = p4.astype(np.float64)

# p1 - p2
A1 = y2 - y1
B1 = x1 - x2
C1 = A1 * x1 + B1 * y1

# p3 - p4
A2 = y4 - y3
B2 = x3 - x4
C2 = A2 * x3 + B2 * y3

det = A1 * B2 - A2 * B1

if abs(det) < 1e-10:
    raise ValueError("Diagonal lines are parallel or ill-conditioned.")

center_x = (B2 * C1 - B1 * C2) / det
center_y = (A1 * C2 - A2 * C1) / det

draw_center_x = int(round(center_x))
draw_center_y = int(round(center_y))

old_center = np.array([[[center_x, center_y]]], dtype=np.float32)

cv2.circle(
    original_display,
    (draw_center_x, draw_center_y),
    4,
    (255, 0, 0),
    -1
)

cv2.putText(
    original_display,
    "CENTER",
    (draw_center_x + 8, draw_center_y),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.2,
    (255, 0, 0),
    1
)

# rectification
canvas_size = 600

dst_points = np.float32([
    [0, 0],
    [599, 0],
    [599, 599],
    [0, 599]
])

H, _ = cv2.findHomography(corners, dst_points)

rectified = cv2.warpPerspective(
    image,
    H,
    (canvas_size, canvas_size)
)

# grid
spacing = 60
dash_length = 10
gap_length = 10

for x in range(0, canvas_size, spacing):
    y = 0
    while y < canvas_size:
        y_end = min(y + dash_length, canvas_size)
        cv2.line(rectified, (x, y), (x, y_end), (0, 0, 0), 1)
        y += dash_length + gap_length

for y in range(0, canvas_size, spacing):
    x = 0
    while x < canvas_size:
        x_end = min(x + dash_length, canvas_size)
        cv2.line(rectified, (x, y), (x_end, y), (0, 0, 0), 1)
        x += dash_length + gap_length

# new center
new_center = cv2.perspectiveTransform(old_center, H)

new_x = int(round(new_center[0][0][0]))
new_y = int(round(new_center[0][0][1]))

# smaller transformed center marker
cv2.circle(
    rectified,
    (new_x, new_y),
    4,
    (0, 0, 255),
    -1
)

cv2.putText(
    rectified,
    "CENTER",
    (new_x + 8, new_y),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.2,
    (0, 0, 255),
    1
)

# display
cv2.imshow("Original Image - Center", original_display)
cv2.imshow("Rectified Image - Grid + Transformed Center", rectified)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [4]:
# task a.4
import cv2
import numpy as np

def segment_discs_and_show(path_to_png):
    """
    Load HSV image from file, segment colored discs, separate touching components.
    Show results and return list of (color_name, u, v) in pixel coordinates.
    """
    # read a3
    rectified_img = cv2.imread(path_to_png, cv2.IMREAD_COLOR)
    if rectified_img is None:
        raise FileNotFoundError(f"Cannot load image: {path_to_png}")

    hsv = cv2.cvtColor(rectified_img, cv2.COLOR_BGR2HSV)
    results = []

    # blue & green: 1 range/ red: 2 ranges
    color_ranges = {
        'red': [
            ([0,   100, 100], [10,  255, 255]),   # low red
            ([170, 100, 100], [180, 255, 255])    # high red
        ],
        'green': [
            ([40,  80,  80],  [80,  255, 255])
        ],
        'blue': [
            ([100, 80, 80],   [130, 255, 255])
        ],
    }


    # kernels morphological separation

    open_kernel      = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    close_kernel     = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    separate_kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))

    # copy of original
    vis_img = rectified_img.copy()

    # process each color 
    for color_name, ranges in color_ranges.items():

        # combine masks for red
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        for lo, hi in ranges:
            lo = np.array(lo, dtype=np.uint8)
            hi = np.array(hi, dtype=np.uint8)
            mask |= cv2.inRange(hsv, lo, hi)

        # morphological separation
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  open_kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, close_kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  separate_kernel)

        # connected components
        n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
            mask, connectivity=8
        )

        # random color for components (displayed only)
        # shows separate images for separate disks on black background
        comp_vis = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)
        rng = np.random.default_rng(42)

        for label in range(1, n_labels):  # skip background
            area = stats[label, cv2.CC_STAT_AREA]
            if area < 400:  # for small noise
                continue

            # centroid
            cx, cy = centroids[label]
            u = int(round(cx))
            v = int(round(cy))

            # store results
            results.append((color_name, u, v))

            # draw centroid
            cv2.circle(vis_img, (u, v), 4, (0, 0, 255), -1)
            cv2.putText(
                vis_img,
                f"{color_name}",
                (u + 5, v - 5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.4,
                (0, 0, 255),
                1
            )

            color = rng.integers(50, 255, size=3, dtype=np.uint8)
            comp_vis[labels == label] = color

        # show per color imgs
        cv2.imshow(f"{color_name} components (separated)", comp_vis)

    # final
    cv2.imshow("Original with centroids", vis_img)

    # tupple out
    tuple_output = [(color, (u, v)) for (color, u, v) in results]
    print("Detected discs (color, (u, v)):")
    for t in tuple_output:
        print(t)

    cv2.waitKey(0)
    cv2.destroyAllWindows()

    return tuple_output


# run
path_to_png = r"C:\Users\Asus\Desktop\hasti\a3.png"   # edit this as needed
results = segment_discs_and_show(path_to_png)


Detected discs (color, (u, v)):
('red', (403, 114))
('red', (248, 198))
('green', (92, 353))
('green', (227, 456))
('blue', (486, 529))


In [5]:
# task a.5 (full pipeline)
import cv2
import numpy as np
from pathlib import Path


GROUND_TRUTH_DISCS = [
    {"color": "red",   "world_xy": (+0.18, -0.10)},
    {"color": "green", "world_xy": (-0.05, +0.20)},
    {"color": "blue",  "world_xy": (-0.22, -0.18)},
    {"color": "red",   "world_xy": (+0.10, +0.05)},
    {"color": "green", "world_xy": (-0.15, +0.07)},
]


def detect_corner_markers(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    _, binary = cv2.threshold(
        hsv[:, :, 2],
        50,
        255,
        cv2.THRESH_BINARY_INV
    )
    
    contours, _ = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )
    
    square_candidates = []
    
    for cnt in contours:
        area = cv2.contourArea(cnt)
        
        if area < 100:
            continue
        
        x, y, w, h = cv2.boundingRect(cnt)
        box_area = w * h
        squareness = area / box_area if box_area > 0 else 0
        
        if squareness > 0.3:
            square_candidates.append(cnt)
    
    square_candidates.sort(key=cv2.contourArea, reverse=True)
    top4 = square_candidates[:4]
    
    centers = []
    for cnt in top4:
        M = cv2.moments(cnt)
        
        if M['m00'] == 0:
            continue
        
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
        centers.append((cx, cy))
    
    centers = sort_clockwise(centers)
    return np.float32(centers)


def sort_clockwise(pts):
    pts = sorted(pts, key=lambda p: p[1])
    top = sorted(pts[:2], key=lambda p: p[0])
    bottom = sorted(pts[2:], key=lambda p: p[0], reverse=True)
    return [top[0], top[1], bottom[0], bottom[1]]


def compute_center(corners):
    p1, p2, p3, p4 = corners[0], corners[2], corners[1], corners[3]
    
    x1, y1 = p1.astype(np.float64)
    x2, y2 = p2.astype(np.float64)
    x3, y3 = p3.astype(np.float64)
    x4, y4 = p4.astype(np.float64)
    
    A1 = y2 - y1
    B1 = x1 - x2
    C1 = A1 * x1 + B1 * y1
    
    A2 = y4 - y3
    B2 = x3 - x4
    C2 = A2 * x3 + B2 * y3
    
    det = A1 * B2 - A2 * B1
    
    center_x = (B2 * C1 - B1 * C2) / det
    center_y = (A1 * C2 - A2 * C1) / det
    
    return np.array([[[center_x, center_y]]], dtype=np.float32)


def rectify_image(image, corners, canvas_size=600):
    dst_points = np.float32([
        [0, 0],
        [canvas_size - 1, 0],
        [canvas_size - 1, canvas_size - 1],
        [0, canvas_size - 1]
    ])
    
    H, _ = cv2.findHomography(corners, dst_points)
    
    rectified = cv2.warpPerspective(
        image,
        H,
        (canvas_size, canvas_size)
    )
    
    return rectified, H


def draw_grid(image, canvas_size=600, grid_spacing=60, dash_length=10, gap_length=10):
    result = image.copy()
    
    for x in range(0, canvas_size, grid_spacing):
        y = 0
        while y < canvas_size:
            y_end = min(y + dash_length, canvas_size)
            cv2.line(result, (x, y), (x, y_end), (0, 0, 0), 1)
            y += dash_length + gap_length
    
    for y in range(0, canvas_size, grid_spacing):
        x = 0
        while x < canvas_size:
            x_end = min(x + dash_length, canvas_size)
            cv2.line(result, (x, y), (x_end, y), (0, 0, 0), 1)
            x += dash_length + gap_length
    
    return result


def segment_discs(rectified_img):
    hsv = cv2.cvtColor(rectified_img, cv2.COLOR_BGR2HSV)
    results = []
    
    color_ranges = {
        'red': [
            ([0,   100, 100], [10,  255, 255]),
            ([170, 100, 100], [180, 255, 255])
        ],
        'green': [
            ([40,  80,  80],  [80,  255, 255])
        ],
        'blue': [
            ([100, 80, 80],   [130, 255, 255])
        ],
    }
    
    open_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    separate_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    
    for color_name, ranges in color_ranges.items():
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        
        for lo, hi in ranges:
            lo = np.array(lo, dtype=np.uint8)
            hi = np.array(hi, dtype=np.uint8)
            mask |= cv2.inRange(hsv, lo, hi)
        
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, open_kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, close_kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, separate_kernel)
        
        n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
            mask, connectivity=8
        )
        
        for label in range(1, n_labels):
            area = stats[label, cv2.CC_STAT_AREA]
            if area < 400:
                continue
            
            cx, cy = centroids[label]
            u = int(round(cx))
            v = int(round(cy))
            
            results.append((color_name, u, v))
    
    return results


def pixel_to_world(u, v, center_u=300, center_v=300, scale=1000):
    world_x = (center_v - v) / scale
    world_y = (center_u - u) / scale
    return (world_x, world_y)


def world_to_pixel(world_x, world_y, center_u=300, center_v=300, scale=1000):
    u = int(center_u - world_y * scale)
    v = int(center_v - world_x * scale)
    return (u, v)


def match_discs_to_ground_truth(detected_discs, ground_truth):
    detected_world = [
        {"color": color, "world_xy": pixel_to_world(u, v), "pixel_uv": (u, v)}
        for color, u, v in detected_discs
    ]
    
    gt_remaining = ground_truth.copy()
    matches = []
    
    for det in detected_world:
        best_match = None
        best_distance = float('inf')
        
        for gt in gt_remaining:
            if gt["color"] != det["color"]:
                continue
            
            dx = det["world_xy"][0] - gt["world_xy"][0]
            dy = det["world_xy"][1] - gt["world_xy"][1]
            distance = np.sqrt(dx**2 + dy**2)
            
            if distance < best_distance:
                best_distance = distance
                best_match = gt
        
        if best_match:
            error_x = det["world_xy"][0] - best_match["world_xy"][0]
            error_y = det["world_xy"][1] - best_match["world_xy"][1]
            
            matches.append({
                "color": det["color"],
                "detected_world_xy": det["world_xy"],
                "detected_pixel_uv": det["pixel_uv"],
                "ground_truth_world_xy": best_match["world_xy"],
                "error_xy": (error_x, error_y),
                "error_magnitude": best_distance
            })
            
            gt_remaining.remove(best_match)
    
    return matches, gt_remaining


def vision_pipeline(image_path, output_dir="./output", canvas_size=600, 
                   grid_spacing=60, visualize=True):
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    image = cv2.imread(str(image_path))
    
    print("="*60)
    print("VISION PIPELINE")
    print("="*60)
    
    print("\nStep 1: Detecting corner markers...")
    corners = detect_corner_markers(image)
    print(f"Detected corners:\n{corners}")
    
    print("\nStep 2: Computing center and rectifying...")
    center_original = compute_center(corners)
    rectified_image, homography = rectify_image(image, corners, canvas_size)
    center_rectified = cv2.perspectiveTransform(center_original, homography)
    
    cx_orig = int(round(center_original[0][0][0]))
    cy_orig = int(round(center_original[0][0][1]))
    cx_rect = int(round(center_rectified[0][0][0]))
    cy_rect = int(round(center_rectified[0][0][1]))
    
    print(f"Original center: ({cx_orig}, {cy_orig})")
    print(f"Rectified center: ({cx_rect}, {cy_rect})")
    
    print("\nCorner world coordinates:")
    corner_labels = ["Top-Left", "Top-Right", "Bottom-Right", "Bottom-Left"]
    expected_coords = [(0.3, 0.3), (0.3, -0.3), (-0.3, -0.3), (-0.3, 0.3)]
    for i, (label, expected) in enumerate(zip(corner_labels, expected_coords)):
        u, v = 0 if i % 2 == 0 else canvas_size - 1, 0 if i < 2 else canvas_size - 1
        world_x, world_y = pixel_to_world(u, v)
        print(f"  {label}: pixel ({u}, {v}) -> world ({world_x:+.3f}, {world_y:+.3f}) [expected: {expected}]")
    
    rectified_with_grid = draw_grid(rectified_image, canvas_size, grid_spacing)
    
    print("\nStep 3: Segmenting colored discs...")
    detected_discs = segment_discs(rectified_image)
    
    print(f"\nDetected {len(detected_discs)} discs (pixel coordinates):")
    for color, u, v in detected_discs:
        print(f"  {color}: ({u}, {v})")
    
    print("\nStep 4: Converting to world coordinates...")
    discs_world = []
    for color, u, v in detected_discs:
        world_x, world_y = pixel_to_world(u, v)
        discs_world.append({
            "color": color,
            "pixel_uv": (u, v),
            "world_xy": (world_x, world_y)
        })
        print(f"  {color}: pixel ({u}, {v}) -> world ({world_x:+.3f}, {world_y:+.3f})")
    
    print("\nStep 5: Comparing with ground truth...")
    matches, unmatched_gt = match_discs_to_ground_truth(detected_discs, GROUND_TRUTH_DISCS)
    
    print(f"\nMatched {len(matches)} discs:")
    print("\n" + "-"*80)
    print(f"{'Color':<8} {'Detected (world)':<20} {'Ground Truth':<20} {'Error':<20} {'Magnitude':<10}")
    print("-"*80)
    
    for match in matches:
        det_str = f"({match['detected_world_xy'][0]:+.3f}, {match['detected_world_xy'][1]:+.3f})"
        gt_str = f"({match['ground_truth_world_xy'][0]:+.3f}, {match['ground_truth_world_xy'][1]:+.3f})"
        err_str = f"({match['error_xy'][0]:+.3f}, {match['error_xy'][1]:+.3f})"
        mag_str = f"{match['error_magnitude']:.4f}"
        
        print(f"{match['color']:<8} {det_str:<20} {gt_str:<20} {err_str:<20} {mag_str:<10}")
    
    if unmatched_gt:
        print(f"\nUnmatched ground truth discs: {len(unmatched_gt)}")
        for gt in unmatched_gt:
            print(f"  {gt['color']}: {gt['world_xy']}")
    
    if visualize:
        vis_corners = image.copy()
        labels = ["TL", "TR", "BR", "BL"]
        for i, (x, y) in enumerate(corners.astype(int)):
            cv2.circle(vis_corners, (x, y), 10, (0, 0, 255), -1)
            cv2.putText(vis_corners, labels[i], (x + 10, y - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        
        vis_center = image.copy()
        cv2.circle(vis_center, (cx_orig, cy_orig), 4, (255, 0, 0), -1)
        cv2.putText(vis_center, "CENTER", (cx_orig + 8, cy_orig),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.2, (255, 0, 0), 1)
        
        vis_rectified = rectified_with_grid.copy()
        cv2.circle(vis_rectified, (cx_rect, cy_rect), 4, (0, 0, 255), -1)
        cv2.putText(vis_rectified, "CENTER", (cx_rect + 8, cy_rect),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.2, (0, 0, 255), 1)
        
        vis_discs = rectified_image.copy()
        for color, u, v in detected_discs:
            cv2.circle(vis_discs, (u, v), 4, (0, 0, 255), -1)
            cv2.putText(vis_discs, color, (u + 5, v - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
        
        vis_comparison = rectified_image.copy()
        
        cv2.circle(vis_comparison, (300, 300), 5, (128, 128, 128), -1)
        
        for gt in GROUND_TRUTH_DISCS:
            gt_u, gt_v = world_to_pixel(gt["world_xy"][0], gt["world_xy"][1])
            cv2.circle(vis_comparison, (gt_u, gt_v), 8, (0, 255, 0), 2)
            cv2.putText(vis_comparison, f"GT-{gt['color']}", (gt_u + 10, gt_v - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 0), 1)
        
        for match in matches:
            det_u, det_v = match['detected_pixel_uv']
            gt_u, gt_v = world_to_pixel(match['ground_truth_world_xy'][0], 
                                        match['ground_truth_world_xy'][1])
            
            cv2.circle(vis_comparison, (det_u, det_v), 6, (0, 0, 255), -1)
            cv2.line(vis_comparison, (det_u, det_v), (gt_u, gt_v), (255, 0, 255), 1)
            
            mid_u = (det_u + gt_u) // 2
            mid_v = (det_v + gt_v) // 2
            cv2.putText(vis_comparison, f"{match['error_magnitude']:.3f}", 
                       (mid_u, mid_v), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (255, 0, 255), 1)
        
        cv2.imwrite(str(output_dir / "1_detected_corners.jpg"), vis_corners)
        cv2.imwrite(str(output_dir / "2_original_center.jpg"), vis_center)
        cv2.imwrite(str(output_dir / "3_rectified_grid.jpg"), vis_rectified)
        cv2.imwrite(str(output_dir / "4_segmented_discs.jpg"), vis_discs)
        cv2.imwrite(str(output_dir / "5_error_comparison.jpg"), vis_comparison)
        cv2.imwrite(str(output_dir / "rectified.png"), rectified_image)
        
        print(f"\nVisualizations saved to {output_dir}/")
    
    errors_x = [m['error_xy'][0] for m in matches]
    errors_y = [m['error_xy'][1] for m in matches]
    errors_mag = [m['error_magnitude'] for m in matches]
    
    return {
        'corners': corners,
        'center_original': (cx_orig, cy_orig),
        'center_rectified': (cx_rect, cy_rect),
        'homography': homography,
        'rectified_image': rectified_image,
        'detected_discs_pixel': [(color, (u, v)) for color, u, v in detected_discs],
        'detected_discs_world': discs_world,
        'matches': matches,
        'unmatched_ground_truth': unmatched_gt,
        'error_statistics': {
            'mean_error_x': np.mean(errors_x) if matches else None,
            'mean_error_y': np.mean(errors_y) if matches else None,
            'rms_error_x': np.sqrt(np.mean(np.array(errors_x)**2)) if matches else None,
            'rms_error_y': np.sqrt(np.mean(np.array(errors_y)**2)) if matches else None,
            'mean_magnitude': np.mean(errors_mag) if matches else None,
            'max_magnitude': np.max(errors_mag) if matches else None,
        }
    }


if __name__ == "__main__":
    image_path = r"C:\Users\Asus\Desktop\hasti\output.png"
    
    results = vision_pipeline(
        image_path=image_path,
        output_dir="./output",
        canvas_size=600,
        grid_spacing=60,
        visualize=True
    )
    
    print("\n" + "="*60)
    print("PIPELINE COMPLETE")
    print("="*60)

VISION PIPELINE

Step 1: Detecting corner markers...
Detected corners:
[[127. 106.]
 [498.  62.]
 [520. 439.]
 [105. 455.]]

Step 2: Computing center and rectifying...
Original center: (304, 256)
Rectified center: (300, 300)

Corner world coordinates:
  Top-Left: pixel (0, 0) -> world (+0.300, +0.300) [expected: (0.3, 0.3)]
  Top-Right: pixel (599, 0) -> world (+0.300, -0.299) [expected: (0.3, -0.3)]
  Bottom-Right: pixel (0, 599) -> world (-0.299, +0.300) [expected: (-0.3, -0.3)]
  Bottom-Left: pixel (599, 599) -> world (-0.299, -0.299) [expected: (-0.3, 0.3)]

Step 3: Segmenting colored discs...

Detected 5 discs (pixel coordinates):
  red: (404, 114)
  red: (248, 197)
  green: (92, 353)
  green: (228, 456)
  blue: (486, 529)

Step 4: Converting to world coordinates...
  red: pixel (404, 114) -> world (+0.186, -0.104)
  red: pixel (248, 197) -> world (+0.103, +0.052)
  green: pixel (92, 353) -> world (-0.053, +0.208)
  green: pixel (228, 456) -> world (-0.156, +0.072)
  blue: pixel (